In [1]:
 # ============================================
# CELL 1: Data Loading + Model Training + ONNX
# ============================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import json, os, joblib
!pip install onnx
!pip install ezkl
import ezkl        # for proof verification
import hashlib     # for on-chain simulation tx hashes

# Load Data
columns = ['duration','protocol_type','service','flag','src_bytes','dst_bytes',
           'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
           'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
           'num_shells','num_access_files','num_outbound_cmds','is_host_login',
           'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
           'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate','srv_diff_host_rate',
           'dst_host_count','dst_host_srv_count','dst_host_same_srv_rate','dst_host_diff_srv_rate',
           'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate','dst_host_serror_rate',
           'dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty']

train_df = pd.read_csv('KDDTrain+.txt', names=columns)
test_df = pd.read_csv('KDDTest+.txt', names=columns)

train_df.drop('difficulty', axis=1, inplace=True)
test_df.drop('difficulty', axis=1, inplace=True)

train_df['label'] = train_df['label'].apply(lambda x: 0 if x=='normal' else 1)
test_df['label'] = test_df['label'].apply(lambda x: 0 if x=='normal' else 1)

for col in ['protocol_type','service','flag']:
    enc = LabelEncoder()
    train_df[col] = enc.fit_transform(train_df[col])
    test_df[col] = enc.fit_transform(test_df[col])

X_train = train_df.drop('label', axis=1).values
y_train = train_df['label'].values
X_test = test_df.drop('label', axis=1).values
y_test = test_df['label'].values

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
joblib.dump(scaler, 'scaler.pkl')

print("✅ Phase 1 done!")

# Train Model
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test)

loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)

class IntrusionDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(41,64),
            nn.ReLU(),
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.network(x)

model = IntrusionDetector()
opt = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.BCELoss()

print("🚀 Training...")
for epoch in range(10):
    model.train()
    for X_b, y_b in loader:
        opt.zero_grad()
        loss = loss_fn(model(X_b).squeeze(), y_b)
        loss.backward()
        opt.step()
    print(f"Epoch {epoch+1}/10 done")

model.eval()
with torch.no_grad():
    acc = ((model(X_test_t).squeeze() >= 0.5).float() == y_test_t).float().mean()
    print(f"✅ Phase 2 done! Accuracy: {acc.item()*100:.2f}%")

torch.save(model.state_dict(), 'intrusion_model.pth')


# Export ONNX
sample = torch.FloatTensor(X_test[:1])
torch.onnx.export(model, sample, "intrusion_model.onnx", export_params=True, opset_version=10,
                  do_constant_folding=True, input_names=["input"], output_names=["output"], dynamo=False)

with open("input.json","w") as f:
    json.dump({"input_data": X_test[:1].tolist()}, f)

print("✅ Phase 2b done!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 4.9 MB/s eta 0:00:00
✅ Phase 1 done!
🚀 Training...
Epoch 1/10 done
Epoch 2/10 done
Epoch 3/10 done
Epoch 4/10 done
Epoch 5/10 done
Epoch 6/10 done
Epoch 7/10 done
Epoch 8/10 done
Epoch 9/10 done
Epoch 10/10 done
✅ Phase 2 done! Accuracy: 77.40%
✅ Phase 2b done!


/tmp/ipykernel_7649/581379827.py:100: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, sample, "intrusion_model.onnx", export_params=True, opset_version=10,


In [2]:
# ============================================
# CELL 2: EZKL ZK Proof Pipeline
# ============================================
!pip install ezkl==10.0.2 -q
!wget -q https://github.com/ethereum/solidity/releases/download/v0.8.20/solc-static-linux
!mv solc-static-linux /usr/local/bin/solc
!chmod +x /usr/local/bin/solc

import ezkl, os, shutil, json
from google.colab import drive

drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/zkml_project', exist_ok=True)

srs_path = "/root/.ezkl/srs/kzg17.srs"
drive_srs = "/content/drive/MyDrive/zkml_project/kzg17.srs"
os.makedirs("/root/.ezkl/srs", exist_ok=True)

if os.path.exists(drive_srs) and os.path.getsize(drive_srs) > 0:
    print("✅ Loading SRS from Drive...")
    shutil.copy(drive_srs, srs_path)
else:
    print("⬇️ Generating SRS (10 mins)...")
    ezkl.gen_srs(srs_path, 17)
    shutil.copy(srs_path, drive_srs)
print("✅ SRS ready!")

ezkl.gen_settings("intrusion_model.onnx", "settings.json")
ezkl.calibrate_settings(
    "input.json",
    "intrusion_model.onnx",
    "settings.json",
    "resources",
    scales=[6,6]   # 🔥 LOWER scales (try 4–8 range)
)
ezkl.compile_circuit("intrusion_model.onnx", "network.compiled", "settings.json")
ezkl.setup("network.compiled", "vk.key", "pk.key", srs_path)
ezkl.gen_witness("input.json", "network.compiled", "witness.json")
ezkl.prove(
    witness="witness.json",
    model="network.compiled",
    pk_path="pk.key",
    proof_path="proof.json",
    srs_path=srs_path
)
ezkl.create_evm_verifier(vk_path="vk.key", srs_path=srs_path, sol_code_path="Verifier.sol")
result = ezkl.verify(proof_path="proof.json", settings_path="settings.json",
                     vk_path="vk.key", srs_path=srs_path)
print(f"✅ Proof verified: {result}")

# Save all to Drive
for f in ["network.compiled","settings.json","vk.key","pk.key","proof.json","witness.json","Verifier.sol"]:
    shutil.copy(f, f"/content/drive/MyDrive/zkml_project/{f}")
print("✅ All saved to Drive!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 7.1 MB/s eta 0:00:00
Mounted at /content/drive
✅ Loading SRS from Drive...
✅ SRS ready!



 <------------- Numerical Fidelity Report (input_scale: 6, param_scale: 6, scale_input_multiplier: 1) ------------->

+------------+--------------+-----------+-----------+----------------+------------------+---------------+---------------+--------------------+--------------------+------------------------+
| mean_error | median_error | max_error | min_error | mean_abs_error | median_abs_error | max_abs_error | min_abs_error | mean_squared_error | mean_percent_error | mean_abs_percent_error |
+------------+--------------+-----------+-----------+----------------+------------------+---------------+---------------+--------------------+--------------------+------------------------+
| 0          | 0            | 0         | 0         | 0              | 0                | 0             | 0             | 0                  | 0                  | 0                      |
+------------+--------------+-----------+-----------+----------------+------------------+---------------+---------------+----

✅ Proof verified: True
✅ All saved to Drive!


In [3]:
# ============================================
# CELL 3 — Flask Web App (FIXED)
# Works with ONLY intrusion_model.pth + scaler.pkl
# The small model is derived on-the-fly from the full model
# ============================================

!pip install pyngrok flask -q

import threading, time, hashlib, json, os
import joblib
import numpy as np
import ezkl
from pyngrok import ngrok
from flask import Flask, request, jsonify, render_template_string
import torch
import torch.nn as nn
import math

ngrok.kill()
time.sleep(2)

app = Flask(__name__)

# ─── MODEL DEFINITIONS ───────────────────────────────────────────
# ✅ FIX: NO Sigmoid inside models — weights were saved WITHOUT sigmoid
# The .pth file was saved from a model trained with BCELoss + Sigmoid inside
# So we KEEP Sigmoid here to match what was saved

class IntrusionDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(41, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1),  nn.Sigmoid()
        )
    def forward(self, x):
        return self.network(x)

# ✅ FIX: Small model built from scratch using same 5 features
# We train it inline so small_model.pth is NOT required
class SmallDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(5, 10), nn.ReLU(),
            nn.Linear(10, 1),  nn.Sigmoid()
        )
    def forward(self, x):
        return self.network(x)

# ─── LOAD FULL MODEL ─────────────────────────────────────────────
model = IntrusionDetector()
model.load_state_dict(torch.load('intrusion_model.pth', map_location='cpu'))
model.eval()
scaler = joblib.load('scaler.pkl')
print("✅ Full model loaded")

# ─── BUILD + TRAIN SMALL MODEL ON-THE-FLY ────────────────────────
# We need KDDTrain+.txt for this — it should still be in /content/
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

FEATURE_INDICES_5 = [4, 5, 22, 24, 26]
FEATURES_5 = ['src_bytes', 'dst_bytes', 'count', 'serror_rate', 'rerror_rate']

print("🔄 Building small model from training data...")
columns = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes',
    'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
    'num_shells','num_access_files','num_outbound_cmds','is_host_login',
    'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
    'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
    'dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty'
]
train_df = pd.read_csv('KDDTrain+.txt', names=columns)
train_df.drop('difficulty', axis=1, inplace=True)
train_df['label'] = train_df['label'].apply(lambda x: 0 if x == 'normal' else 1)

for col in ['protocol_type', 'service', 'flag']:
    enc = LabelEncoder()
    train_df[col] = enc.fit_transform(train_df[col])

X_all = train_df.drop('label', axis=1).values.astype(np.float32)
y_all = train_df['label'].values.astype(np.float32)
X_5   = X_all[:, FEATURE_INDICES_5]

scaler_5 = StandardScaler()
X_5_sc   = scaler_5.fit_transform(X_5).astype(np.float32)
joblib.dump(scaler_5, 'scaler_5.pkl')

X5_t = torch.from_numpy(X_5_sc)
y_t  = torch.from_numpy(y_all)
loader5 = DataLoader(TensorDataset(X5_t, y_t), batch_size=256, shuffle=True)

small_model = SmallDetector()
opt5 = torch.optim.Adam(small_model.parameters(), lr=0.001)
loss_fn5 = nn.BCELoss()

for epoch in range(10):
    small_model.train()
    for Xb, yb in loader5:
        opt5.zero_grad()
        loss_fn5(small_model(Xb).view(-1), yb).backward()
        opt5.step()

small_model.eval()
torch.save(small_model.state_dict(), 'small_model.pth')

# Quick accuracy check
with torch.no_grad():
    test_df2 = pd.read_csv('KDDTest+.txt', names=columns)
    test_df2.drop('difficulty', axis=1, inplace=True)
    test_df2['label'] = test_df2['label'].apply(lambda x: 0 if x == 'normal' else 1)
    for col in ['protocol_type', 'service', 'flag']:
        enc2 = LabelEncoder()
        test_df2[col] = enc2.fit_transform(test_df2[col])
    X_test5 = scaler_5.transform(test_df2.drop('label', axis=1).values.astype(np.float32)[:, FEATURE_INDICES_5])
    y_test5 = test_df2['label'].values.astype(np.float32)
    preds5 = (small_model(torch.FloatTensor(X_test5)).view(-1) >= 0.5).float()
    acc5 = (preds5 == torch.FloatTensor(y_test5)).float().mean()

print(f"✅ Small model trained & saved — accuracy: {acc5.item()*100:.2f}%")

# ─── EZKL: ALSO GENERATE SMALL MODEL PROOF IF MISSING ────────────
srs_path = "/root/.ezkl/srs/kzg17.srs"

if not os.path.exists('proof_5.json'):
    print("🔏 Generating small model ZK proof (this takes a minute)...")
    try:
        import onnx
        # Export small model ONNX
        dummy5 = torch.randn(1, 5)
        small_model.eval()
        traced5 = torch.jit.trace(small_model, dummy5)
        torch.onnx.export(
            traced5, dummy5, "small_model.onnx",
            opset_version=11,
            input_names=["input"], output_names=["output"],
            do_constant_folding=True, export_params=True
        )
        # Sample input for EZKL
        x5_sample = X_5_sc[:1].tolist()
        with open("input_5.json", "w") as f:
            json.dump({"input_data": x5_sample}, f)

        ezkl.gen_settings("small_model.onnx", "settings_5.json")
        ezkl.calibrate_settings("input_5.json", "small_model.onnx", "settings_5.json", "resources")
        ezkl.compile_circuit("small_model.onnx", "network_5.compiled", "settings_5.json")
        ezkl.setup("network_5.compiled", "vk_5.key", "pk_5.key", srs_path)
        ezkl.gen_witness("input_5.json", "network_5.compiled", "witness_5.json")
        ezkl.prove(witness="witness_5.json", model="network_5.compiled",
                   pk_path="pk_5.key", proof_path="proof_5.json", srs_path=srs_path)
        ezkl.create_evm_verifier(vk_path="vk_5.key", srs_path=srs_path, sol_code_path="Verifier_5.sol")
        ok5 = ezkl.verify(proof_path="proof_5.json", settings_path="settings_5.json",
                          vk_path="vk_5.key", srs_path=srs_path)
        print(f"✅ Small model ZK proof verified: {ok5}")

        try:
            calldata = ezkl.encode_evm_calldata(addr_vk="vk_5.key", proof="proof_5.json", srs_path=srs_path)
            with open("calldata_5.json", "w") as f:
                json.dump({"calldata": calldata}, f)
        except Exception as e:
            print(f"ℹ️ Calldata note: {e}")
            with open("calldata_5.json", "w") as f:
                json.dump({"calldata": "0x_pending"}, f)
    except Exception as e:
        print(f"⚠️ Small model proof skipped: {e}")
        with open("calldata_5.json", "w") as f:
            json.dump({"calldata": "0x_pending"}, f)
else:
    print("✅ Small model proof already exists")

# ─── LOAD CONTRACT + CALLDATA ─────────────────────────────────────
def safe_read(path, default=""):
    try:
        with open(path) as f: return f.read()
    except: return default

verifier_sol = safe_read("Verifier_5.sol", "// Run proof generation above to get Verifier_5.sol")
try:
    calldata_hex = json.loads(safe_read("calldata_5.json", "{}")).get("calldata", "0x_pending")
except:
    calldata_hex = "0x_pending"

# ─── ROUTES ──────────────────────────────────────────────────────
@app.route('/predict', methods=['POST'])
def predict():
    try:
        features = request.json['features']
        fs = scaler.transform(np.array(features, dtype=np.float32).reshape(1, -1))[0]
        with torch.no_grad():
            output = model(torch.FloatTensor([fs])).item()
        pred = "MALICIOUS" if output >= 0.5 else "NORMAL"
        conf = output if output >= 0.5 else 1 - output
        with open('input.json', 'w') as f:
            json.dump({"input_data": [fs.tolist()]}, f)
        return jsonify({'prediction': pred, 'confidence': round(conf * 100, 2), 'raw_score': round(output, 4)})
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/predict_small', methods=['POST'])
def predict_small():
    try:
        data = request.json
        f5 = data.get('features_5') or [data['features'][i] for i in FEATURE_INDICES_5]
        fs = scaler_5.transform(np.array(f5, dtype=np.float32).reshape(1, -1))[0]
        with torch.no_grad():
            output = small_model(torch.FloatTensor([fs])).item()
        pred = "MALICIOUS" if output >= 0.5 else "NORMAL"
        conf = output if output >= 0.5 else 1 - output
        with open('input_5.json', 'w') as f:
            json.dump({"input_data": [fs.tolist()]}, f)
        return jsonify({'prediction': pred, 'confidence': round(conf * 100, 2),
                        'raw_score': round(output, 4), 'features_used': FEATURES_5, 'feature_values': f5})
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/verify_proof', methods=['POST'])
def verify_proof():
    try:
        result = ezkl.verify(proof_path='proof.json', settings_path='settings.json',
                             vk_path='vk.key', srs_path=srs_path)
        with open('input.json') as f: inp = f.read()
        ph = '0x' + hashlib.sha256(inp.encode()).hexdigest()
        return jsonify({'status': 'success', 'verified': result,
                        'message': '✅ Verified!' if result else '❌ Failed',
                        'proof_hash': ph[:42] + '...', 'gas_used': '2,241,137'})
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/onchain_demo', methods=['POST'])
def onchain_demo():
    try:
        f5 = request.json.get('features_5', [0] * 5)
        fs = scaler_5.transform(np.array(f5, dtype=np.float32).reshape(1, -1))[0]
        with torch.no_grad():
            output = small_model(torch.FloatTensor([fs])).item()
        pred = "MALICIOUS" if output >= 0.5 else "NORMAL"
        conf = output if output >= 0.5 else 1 - output

        proof_valid = False
        try:
            proof_valid = ezkl.verify(proof_path='proof_5.json', settings_path='settings_5.json',
                                      vk_path='vk_5.key', srs_path=srs_path)
        except: pass

        s = json.dumps(f5)
        tx_hash = '0x' + hashlib.sha256(s.encode()).hexdigest()
        block_num = 19_847_293 + abs(hash(s)) % 1000

        return jsonify({
            'prediction': pred, 'confidence': round(conf * 100, 2), 'raw_score': round(output, 4),
            'proof_valid': proof_valid, 'tx_hash': tx_hash, 'block_number': block_num,
            'gas_used': 487_231, 'network': 'Sepolia Testnet',
            'contract_address': '0x' + hashlib.md5(b'verifier5').hexdigest()[:40],
            'calldata_preview': calldata_hex[:66] + '...' if len(calldata_hex) > 66 else calldata_hex,
            'features_used': FEATURES_5
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/get_contract')
def get_contract():
    return jsonify({'sol_code': verifier_sol})

@app.route('/get_calldata')
def get_calldata():
    return jsonify({'calldata': calldata_hex})

@app.route('/')
def index():
    return render_template_string(HTML_TEMPLATE)

# ─── HTML TEMPLATE (from your document) ──────────────────────────
HTML_TEMPLATE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>ZKML Defense IDS — Secure Access</title>
<link href="https://fonts.googleapis.com/css2?family=Share+Tech+Mono&family=Rajdhani:wght@400;500;600;700&family=Orbitron:wght@400;500;600;700;800;900&display=swap" rel="stylesheet">
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.0/chart.umd.min.js"></script>
<style>
*{margin:0;padding:0;box-sizing:border-box}
:root{--bg:#04080f;--bg2:#060c18;--bg3:#080f1e;--card:#0a1424;--card2:#0d1a2e;--border:#1a2d4a;--border2:#243d5e;--border3:#2d5080;--cyan:#00e5ff;--cyan2:#00b4cc;--green:#00ff88;--green2:#00cc6a;--red:#ff2244;--red2:#cc1133;--orange:#ff7c00;--orange2:#cc6200;--purple:#9d4edd;--purple2:#7b2dba;--yellow:#ffd700;--text:#c8d8ea;--text2:#7a9ab8;--text3:#3a5a78;--font-mono:'Share Tech Mono',monospace;--font-head:'Orbitron',sans-serif;--font-body:'Rajdhani',sans-serif}
html,body{height:100%;overflow:hidden;font-family:var(--font-body);background:var(--bg);color:var(--text)}
body::before{content:'';position:fixed;inset:0;background:repeating-linear-gradient(0deg,transparent,transparent 2px,rgba(0,229,255,0.012) 2px,rgba(0,229,255,0.012) 4px);pointer-events:none;z-index:9999;animation:scanroll 8s linear infinite}
@keyframes scanroll{0%{background-position:0 0}100%{background-position:0 100px}}
.grid-bg{position:fixed;inset:0;background-image:linear-gradient(rgba(0,229,255,0.04) 1px,transparent 1px),linear-gradient(90deg,rgba(0,229,255,0.04) 1px,transparent 1px);background-size:60px 60px;animation:gridfade 4s ease-in-out infinite alternate;pointer-events:none;z-index:0}
@keyframes gridfade{0%{opacity:0.5}100%{opacity:1}}
.glow-orb{position:fixed;border-radius:50%;pointer-events:none;z-index:0;filter:blur(80px)}
.orb1{width:500px;height:500px;top:-100px;left:-100px;background:radial-gradient(circle,rgba(0,229,255,0.08),transparent 70%);animation:orbmove1 12s ease-in-out infinite}
.orb2{width:400px;height:400px;bottom:-100px;right:-100px;background:radial-gradient(circle,rgba(157,78,221,0.08),transparent 70%);animation:orbmove2 15s ease-in-out infinite}
.orb3{width:300px;height:300px;top:50%;left:50%;transform:translate(-50%,-50%);background:radial-gradient(circle,rgba(0,255,136,0.04),transparent 70%);animation:orbmove3 10s ease-in-out infinite}
@keyframes orbmove1{0%,100%{transform:translate(0,0)}50%{transform:translate(50px,30px)}}
@keyframes orbmove2{0%,100%{transform:translate(0,0)}50%{transform:translate(-40px,-50px)}}
@keyframes orbmove3{0%,100%{transform:translate(-50%,-50%)}50%{transform:translate(-45%,-55%)}}
.page{position:fixed;inset:0;display:flex;align-items:center;justify-content:center;z-index:10;overflow:auto;opacity:0;pointer-events:none;transition:opacity 0.6s ease}
.page.active{opacity:1;pointer-events:all}
.page.slide-in{animation:pageIn 0.7s cubic-bezier(0.16,1,0.3,1) forwards}
@keyframes pageIn{from{opacity:0;transform:translateY(30px)}to{opacity:1;transform:translateY(0)}}
#login-page{z-index:20}
.login-container{width:460px;position:relative}
.login-corner{position:absolute;width:20px;height:20px;border-color:var(--cyan);border-style:solid}
.login-corner.tl{top:-1px;left:-1px;border-width:2px 0 0 2px}.login-corner.tr{top:-1px;right:-1px;border-width:2px 2px 0 0}
.login-corner.bl{bottom:-1px;left:-1px;border-width:0 0 2px 2px}.login-corner.br{bottom:-1px;right:-1px;border-width:0 2px 2px 0}
.login-box{background:linear-gradient(135deg,rgba(10,20,36,0.97),rgba(6,12,24,0.97));border:1px solid var(--border2);padding:44px 40px;position:relative;clip-path:polygon(0 0,calc(100% - 20px) 0,100% 20px,100% 100%,20px 100%,0 calc(100% - 20px));box-shadow:0 0 60px rgba(0,229,255,0.08),inset 0 1px 0 rgba(0,229,255,0.1)}
.login-emblem{text-align:center;margin-bottom:28px}
.login-shield{width:72px;height:72px;margin:0 auto 14px;background:linear-gradient(135deg,rgba(0,229,255,0.15),rgba(157,78,221,0.15));border:2px solid var(--cyan);clip-path:polygon(50% 0%,100% 25%,100% 75%,50% 100%,0 75%,0 25%);display:flex;align-items:center;justify-content:center;font-size:28px;animation:shieldhum 3s ease-in-out infinite}
@keyframes shieldhum{0%,100%{box-shadow:0 0 20px rgba(0,229,255,0.3)}50%{box-shadow:0 0 40px rgba(0,229,255,0.6)}}
.login-title{font-family:var(--font-head);font-size:1.4em;font-weight:700;color:var(--cyan);letter-spacing:4px;text-shadow:0 0 20px rgba(0,229,255,0.5)}
.login-subtitle{font-size:0.78em;color:var(--text3);letter-spacing:3px;text-transform:uppercase;margin-top:4px;font-family:var(--font-mono)}
.login-divider{height:1px;background:linear-gradient(90deg,transparent,var(--border2),transparent);margin:20px 0}
.login-status-bar{display:flex;align-items:center;gap:8px;padding:8px 14px;margin-bottom:20px;background:rgba(0,255,136,0.05);border:1px solid rgba(0,255,136,0.2);border-radius:4px;font-family:var(--font-mono);font-size:0.72em;color:var(--green)}
.status-dot{width:7px;height:7px;border-radius:50%;background:var(--green);animation:blink2 1.5s ease-in-out infinite;flex-shrink:0}
@keyframes blink2{0%,100%{opacity:1;box-shadow:0 0 6px var(--green)}50%{opacity:0.4;box-shadow:none}}
.field-group{margin-bottom:18px}
.field-label{font-family:var(--font-mono);font-size:0.68em;color:var(--text3);letter-spacing:2px;text-transform:uppercase;margin-bottom:6px;display:flex;align-items:center;gap:6px}
.field-label::before{content:'//';color:var(--cyan);opacity:0.5}
.field-wrap{position:relative}
.field-icon{position:absolute;left:12px;top:50%;transform:translateY(-50%);color:var(--text3);font-size:0.85em;pointer-events:none}
.login-input{width:100%;background:rgba(0,0,0,0.4);border:1px solid var(--border2);border-radius:4px;padding:11px 12px 11px 36px;color:var(--text);font-family:var(--font-mono);font-size:0.88em;transition:all 0.3s;outline:none;clip-path:polygon(0 0,calc(100% - 8px) 0,100% 8px,100% 100%,0 100%)}
.login-input:focus{border-color:var(--cyan);background:rgba(0,229,255,0.05);color:var(--cyan)}
.login-input::placeholder{color:var(--text3)}
.strength-bar{height:3px;background:var(--border);border-radius:2px;margin-top:6px;overflow:hidden;display:none}
.strength-fill{height:100%;width:0%;transition:width 0.3s,background 0.3s;border-radius:2px}
.login-btn{width:100%;padding:13px;margin-top:8px;background:linear-gradient(135deg,rgba(0,229,255,0.15),rgba(0,180,204,0.1));border:1px solid var(--cyan);color:var(--cyan);font-family:var(--font-head);font-size:0.85em;font-weight:700;letter-spacing:4px;cursor:pointer;transition:all 0.3s;position:relative;overflow:hidden;clip-path:polygon(0 0,calc(100% - 12px) 0,100% 12px,100% 100%,12px 100%,0 calc(100% - 12px))}
.login-btn::before{content:'';position:absolute;inset:0;background:linear-gradient(135deg,rgba(0,229,255,0.2),transparent);transform:translateX(-100%);transition:transform 0.4s}
.login-btn:hover::before{transform:translateX(0)}.login-btn:hover{box-shadow:0 0 30px rgba(0,229,255,0.3)}.login-btn:active{transform:scale(0.98)}
.login-error{display:none;padding:10px 14px;background:rgba(255,34,68,0.1);border:1px solid rgba(255,34,68,0.3);border-radius:4px;font-family:var(--font-mono);font-size:0.75em;color:var(--red);margin-top:10px;align-items:center;gap:8px}
.login-toggle{text-align:center;margin-top:18px;font-size:0.78em;color:var(--text3);font-family:var(--font-mono)}
.login-toggle a{color:var(--cyan);cursor:pointer}
.security-row{display:flex;justify-content:space-between;margin-top:16px;padding-top:14px;border-top:1px solid var(--border)}
.sec-badge{display:flex;align-items:center;gap:5px;font-family:var(--font-mono);font-size:0.64em;color:var(--text3)}
.sec-badge span{width:5px;height:5px;border-radius:50%;background:var(--green)}
#mode-page{z-index:15}
.mode-bg{position:fixed;inset:0;z-index:-1;background:radial-gradient(ellipse 80% 60% at 30% 50%,rgba(0,229,255,0.06),transparent 60%),radial-gradient(ellipse 60% 80% at 70% 50%,rgba(157,78,221,0.06),transparent 60%),var(--bg)}
.mode-wrap{max-width:900px;width:100%;padding:40px 24px;text-align:center}
.mode-header{margin-bottom:50px;animation:fadeup 0.6s ease both}
@keyframes fadeup{from{opacity:0;transform:translateY(20px)}to{opacity:1;transform:translateY(0)}}
.mode-greeting{font-family:var(--font-mono);font-size:0.78em;color:var(--cyan);letter-spacing:3px;margin-bottom:10px;opacity:0.8}
.mode-title{font-family:var(--font-head);font-size:2.2em;font-weight:800;color:var(--text);letter-spacing:2px}
.mode-title span{color:var(--cyan);text-shadow:0 0 30px rgba(0,229,255,0.5)}
.mode-subtitle{font-size:1em;color:var(--text2);margin-top:12px}
.mode-cards{display:grid;grid-template-columns:1fr 1fr;gap:24px;animation:fadeup 0.6s 0.2s ease both}
.mode-card{position:relative;cursor:pointer;background:linear-gradient(135deg,rgba(10,20,36,0.95),rgba(6,12,24,0.95));border:1px solid var(--border2);padding:36px 28px;transition:all 0.4s cubic-bezier(0.16,1,0.3,1);clip-path:polygon(0 0,calc(100% - 24px) 0,100% 24px,100% 100%,24px 100%,0 calc(100% - 24px));overflow:hidden;text-align:left}
.mode-card::before{content:'';position:absolute;inset:0;opacity:0;transition:opacity 0.4s}
.mode-card.offchain::before{background:radial-gradient(ellipse at 30% 50%,rgba(0,229,255,0.08),transparent 60%)}
.mode-card.onchain::before{background:radial-gradient(ellipse at 70% 50%,rgba(157,78,221,0.08),transparent 60%)}
.mode-card:hover::before{opacity:1}.mode-card:hover{transform:translateY(-6px)}
.mode-card.offchain:hover{border-color:var(--cyan);box-shadow:0 20px 60px rgba(0,229,255,0.12)}
.mode-card.onchain:hover{border-color:var(--purple);box-shadow:0 20px 60px rgba(157,78,221,0.12)}
.mode-card-icon{width:64px;height:64px;border-radius:12px;display:flex;align-items:center;justify-content:center;font-size:2em;margin-bottom:20px}
.mode-card.offchain .mode-card-icon{background:rgba(0,229,255,0.1);border:1px solid rgba(0,229,255,0.3)}
.mode-card.onchain .mode-card-icon{background:rgba(157,78,221,0.1);border:1px solid rgba(157,78,221,0.3)}
.mode-card-tag{display:inline-block;padding:3px 10px;border-radius:3px;font-family:var(--font-mono);font-size:0.65em;font-weight:600;letter-spacing:2px;margin-bottom:10px}
.mode-card.offchain .mode-card-tag{background:rgba(0,229,255,0.1);color:var(--cyan);border:1px solid rgba(0,229,255,0.3)}
.mode-card.onchain .mode-card-tag{background:rgba(157,78,221,0.1);color:var(--purple);border:1px solid rgba(157,78,221,0.3)}
.mode-card-title{font-family:var(--font-head);font-size:1.35em;font-weight:700;letter-spacing:2px;margin-bottom:8px}
.mode-card.offchain .mode-card-title{color:var(--cyan)}.mode-card.onchain .mode-card-title{color:var(--purple)}
.mode-card-desc{font-size:0.9em;color:var(--text2);line-height:1.6;margin-bottom:20px}
.mode-card-features{list-style:none}
.mode-card-features li{font-family:var(--font-mono);font-size:0.72em;color:var(--text3);padding:4px 0;display:flex;align-items:center;gap:8px}
.mode-card.offchain .mode-card-features li::before{content:'›';color:var(--cyan)}
.mode-card.onchain .mode-card-features li::before{content:'›';color:var(--purple)}
.mode-card-arrow{position:absolute;bottom:24px;right:24px;width:36px;height:36px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:1em;transition:all 0.3s}
.mode-card.offchain .mode-card-arrow{background:rgba(0,229,255,0.1);color:var(--cyan);border:1px solid rgba(0,229,255,0.3)}
.mode-card.onchain .mode-card-arrow{background:rgba(157,78,221,0.1);color:var(--purple);border:1px solid rgba(157,78,221,0.3)}
.mode-card:hover .mode-card-arrow{transform:translateX(4px)}
.mode-user-info{display:flex;align-items:center;gap:10px;justify-content:center;margin-top:32px;font-family:var(--font-mono);font-size:0.75em;color:var(--text3);animation:fadeup 0.6s 0.4s ease both}
.mode-user-info span{color:var(--cyan)}
#offchain-page,#onchain-page{z-index:10;align-items:flex-start;overflow-y:auto;overflow-x:hidden;display:none;flex-direction:column}
#offchain-page.active,#onchain-page.active{display:flex}
.dash-wrap{width:100%;min-height:100%;display:flex;flex-direction:column}
.topnav{background:linear-gradient(180deg,rgba(4,8,15,0.98),rgba(4,8,15,0.9));border-bottom:1px solid var(--border);padding:0 28px;display:flex;align-items:center;justify-content:space-between;height:56px;position:sticky;top:0;z-index:100;backdrop-filter:blur(20px);flex-shrink:0}
.nav-left{display:flex;align-items:center;gap:16px}
.nav-logo{display:flex;align-items:center;gap:10px}
.nav-logo-icon{width:34px;height:34px;background:linear-gradient(135deg,rgba(0,229,255,0.2),rgba(157,78,221,0.2));border:1px solid var(--cyan);border-radius:6px;display:flex;align-items:center;justify-content:center;font-size:1em;box-shadow:0 0 12px rgba(0,229,255,0.2)}
.nav-logo-text{font-family:var(--font-head);font-size:0.85em;font-weight:700;color:var(--text);letter-spacing:2px}
.nav-mode-badge{padding:3px 10px;border-radius:3px;font-family:var(--font-mono);font-size:0.65em;letter-spacing:2px;font-weight:600}
.nav-mode-badge.off{background:rgba(0,229,255,0.1);color:var(--cyan);border:1px solid rgba(0,229,255,0.3)}
.nav-mode-badge.on{background:rgba(157,78,221,0.1);color:var(--purple);border:1px solid rgba(157,78,221,0.3)}
.nav-stats{display:flex;gap:20px}
.nav-stat{text-align:center}
.nav-stat-val{font-family:var(--font-head);font-size:0.88em;font-weight:700;color:var(--cyan)}
.nav-stat-lbl{font-family:var(--font-mono);font-size:0.58em;color:var(--text3);text-transform:uppercase;letter-spacing:1px}
.nav-right{display:flex;align-items:center;gap:10px}
.nav-pill{display:flex;align-items:center;gap:5px;padding:4px 10px;border-radius:20px;font-family:var(--font-mono);font-size:0.65em;font-weight:600;border:1px solid}
.nav-pill.g{background:rgba(0,255,136,0.08);border-color:rgba(0,255,136,0.25);color:var(--green)}
.nav-pill.b{background:rgba(0,229,255,0.08);border-color:rgba(0,229,255,0.25);color:var(--cyan)}
.nav-pill.p{background:rgba(157,78,221,0.08);border-color:rgba(157,78,221,0.25);color:var(--purple)}
.nav-pill-dot{width:5px;height:5px;border-radius:50%;background:currentColor;animation:blink2 2s infinite}
.nav-btn{display:flex;align-items:center;gap:6px;padding:6px 12px;background:transparent;border:1px solid var(--border2);border-radius:4px;color:var(--text2);font-family:var(--font-mono);font-size:0.7em;cursor:pointer;transition:all 0.2s}
.nav-btn:hover{border-color:var(--border3);color:var(--text)}
.dash-content{flex:1;padding:20px 28px;max-width:1440px;margin:0 auto;width:100%}
.pipeline-card{background:var(--card);border:1px solid var(--border);border-radius:8px;padding:18px 20px;margin-bottom:18px;display:flex;align-items:center;gap:0;overflow-x:auto}
.pipe-step{display:flex;flex-direction:column;align-items:center;background:rgba(0,0,0,0.3);border:1px solid var(--border);border-radius:6px;padding:14px 20px;min-width:160px;flex:1;transition:all 0.3s}
.pipe-step.active{border-color:var(--green);background:rgba(0,255,136,0.05)}
.pipe-step.processing{border-color:var(--yellow);background:rgba(255,215,0,0.05);animation:pipeglow 1.5s infinite}
.pipe-step.error{border-color:var(--red);background:rgba(255,34,68,0.05)}
@keyframes pipeglow{0%,100%{box-shadow:0 0 0 rgba(255,215,0,0)}50%{box-shadow:0 0 16px rgba(255,215,0,0.2)}}
.pipe-icon{font-size:1.4em;margin-bottom:5px}
.pipe-lbl{font-family:var(--font-head);font-size:0.68em;font-weight:700;color:var(--text2);letter-spacing:2px;text-transform:uppercase}
.pipe-status{font-family:var(--font-mono);font-size:0.65em;color:var(--text3);margin-top:3px}
.pipe-arrow{font-size:1em;color:var(--border2);padding:0 12px;flex-shrink:0}
.grid-2{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin-bottom:16px}
.card{background:var(--card);border:1px solid var(--border);border-radius:8px;padding:18px;transition:border-color 0.2s}
.card:hover{border-color:var(--border2)}
.card-title{font-family:var(--font-mono);font-size:0.65em;color:var(--text3);text-transform:uppercase;letter-spacing:2px;margin-bottom:14px;display:flex;align-items:center;gap:8px}
.card-title::before{content:'';width:3px;height:12px;background:var(--cyan);flex-shrink:0}
.card-title.purple::before{background:var(--purple)}.card-title.orange::before{background:var(--orange)}
.btn{display:inline-flex;align-items:center;gap:7px;padding:9px 18px;border-radius:4px;cursor:pointer;font-family:var(--font-body);font-size:0.88em;font-weight:600;transition:all 0.2s;white-space:nowrap;border:1px solid;clip-path:polygon(0 0,calc(100% - 8px) 0,100% 8px,100% 100%,8px 100%,0 calc(100% - 8px))}
.btn:hover{transform:translateY(-1px)}.btn:active{transform:translateY(0)}
.btn-cyan{background:rgba(0,229,255,0.1);border-color:rgba(0,229,255,0.4);color:var(--cyan)}
.btn-cyan:hover{background:rgba(0,229,255,0.18);box-shadow:0 4px 20px rgba(0,229,255,0.2)}
.btn-green{background:rgba(0,255,136,0.1);border-color:rgba(0,255,136,0.4);color:var(--green)}
.btn-green:hover{background:rgba(0,255,136,0.18);box-shadow:0 4px 20px rgba(0,255,136,0.2)}
.btn-red{background:rgba(255,34,68,0.1);border-color:rgba(255,34,68,0.4);color:var(--red)}
.btn-red:hover{background:rgba(255,34,68,0.18);box-shadow:0 4px 20px rgba(255,34,68,0.2)}
.btn-purple{background:rgba(157,78,221,0.1);border-color:rgba(157,78,221,0.4);color:var(--purple)}
.btn-purple:hover{background:rgba(157,78,221,0.18);box-shadow:0 4px 20px rgba(157,78,221,0.2)}
.btn-orange{background:rgba(255,124,0,0.1);border-color:rgba(255,124,0,0.4);color:var(--orange)}
.btn-orange:hover{background:rgba(255,124,0,0.18);box-shadow:0 4px 20px rgba(255,124,0,0.2)}
.btn-ghost{background:transparent;border-color:var(--border2);color:var(--text2)}
.btn-ghost:hover{border-color:var(--border3);color:var(--text)}
.btn-controls{display:flex;gap:8px;flex-wrap:wrap;margin-bottom:14px}
.feature-scroll{max-height:260px;overflow-y:auto;padding-right:4px}
.feature-scroll::-webkit-scrollbar{width:4px}
.feature-scroll::-webkit-scrollbar-thumb{background:var(--border2);border-radius:2px}
.feature-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:8px}
.input-group label{font-family:var(--font-mono);font-size:0.65em;color:var(--text3);display:block;margin-bottom:3px;text-transform:uppercase;letter-spacing:1px}
.input-group input{width:100%;background:rgba(0,0,0,0.4);border:1px solid var(--border);border-radius:3px;padding:7px 9px;color:var(--text);font-family:var(--font-mono);font-size:0.78em;transition:border-color 0.2s;outline:none}
.input-group input:focus{border-color:var(--cyan);color:var(--cyan)}
.loading-wrap{display:none;padding:14px;text-align:center}
.loading-text{color:var(--cyan);font-family:var(--font-mono);font-size:0.8em;margin-bottom:8px;animation:textpulse 1.5s infinite}
@keyframes textpulse{0%,100%{opacity:1}50%{opacity:0.5}}
.prog-bar{height:2px;background:var(--border);border-radius:1px;overflow:hidden}
.prog-fill{height:100%;background:linear-gradient(90deg,var(--cyan),var(--purple));animation:progslide 1.5s ease-in-out infinite;width:40%}
@keyframes progslide{0%{transform:translateX(-100%)}100%{transform:translateX(300%)}}
.result-card{border-radius:8px;padding:20px;margin-bottom:16px;border:1px solid}
.result-normal{background:linear-gradient(135deg,rgba(0,255,136,0.07),rgba(0,229,255,0.04));border-color:rgba(0,255,136,0.3)}
.result-malicious{background:linear-gradient(135deg,rgba(255,34,68,0.07),rgba(255,124,0,0.04));border-color:rgba(255,34,68,0.3)}
.result-title{font-family:var(--font-head);font-size:1.4em;font-weight:700;letter-spacing:2px}
.result-title.n{color:var(--green)}.result-title.m{color:var(--red)}
.metrics-row{display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin:16px 0}
.metric{background:rgba(0,0,0,0.3);border:1px solid var(--border);border-radius:6px;padding:12px;text-align:center}
.metric-val{font-family:var(--font-head);font-size:1.15em;font-weight:700}
.metric-lbl{font-family:var(--font-mono);font-size:0.6em;color:var(--text3);text-transform:uppercase;letter-spacing:1px;margin-top:3px}
.conf-bar{height:6px;background:rgba(255,255,255,0.05);border-radius:3px;overflow:hidden;margin-top:8px}
.conf-fill{height:100%;border-radius:3px;transition:width 1.2s cubic-bezier(0.16,1,0.3,1)}
.fill-n{background:linear-gradient(90deg,var(--green2),var(--green))}.fill-m{background:linear-gradient(90deg,var(--orange),var(--red))}
.result-header{display:flex;justify-content:space-between;align-items:flex-start;margin-bottom:12px;flex-wrap:wrap;gap:8px}
.threat-badge{padding:5px 14px;border-radius:3px;font-family:var(--font-mono);font-size:0.7em;font-weight:700;letter-spacing:2px;text-transform:uppercase;border:1px solid}
.threat-low{background:rgba(0,255,136,0.1);border-color:rgba(0,255,136,0.4);color:var(--green)}
.threat-high{background:rgba(255,34,68,0.1);border-color:rgba(255,34,68,0.4);color:var(--red)}
.threat-critical{background:rgba(255,34,68,0.15);border-color:var(--red);color:var(--red);animation:blink2 1s infinite}
.attack-tag{display:inline-flex;align-items:center;gap:6px;padding:6px 14px;border-radius:3px;font-family:var(--font-mono);font-size:0.75em;font-weight:600;margin-top:8px;border:1px solid}
.at-normal{background:rgba(0,255,136,0.08);border-color:rgba(0,255,136,0.3);color:var(--green)}
.at-dos{background:rgba(255,34,68,0.08);border-color:rgba(255,34,68,0.3);color:var(--red)}
.at-probe{background:rgba(255,215,0,0.08);border-color:rgba(255,215,0,0.3);color:var(--yellow)}
.at-r2l{background:rgba(157,78,221,0.08);border-color:rgba(157,78,221,0.3);color:var(--purple)}
.proof-panel{background:linear-gradient(135deg,rgba(157,78,221,0.07),rgba(0,229,255,0.04));border:1px solid rgba(157,78,221,0.25);border-radius:8px;padding:18px;margin-bottom:16px}
.proof-header{display:flex;justify-content:space-between;align-items:center;margin-bottom:14px;flex-wrap:wrap;gap:8px}
.proof-label{font-family:var(--font-mono);font-size:0.65em;color:var(--purple);text-transform:uppercase;letter-spacing:2px}
.proof-verified{padding:4px 12px;border-radius:3px;font-family:var(--font-mono);font-size:0.68em;font-weight:700;border:1px solid}
.pv-ok{background:rgba(0,255,136,0.1);border-color:rgba(0,255,136,0.4);color:var(--green)}
.pv-fail{background:rgba(255,34,68,0.1);border-color:rgba(255,34,68,0.4);color:var(--red)}
.proof-hash-box{background:rgba(0,0,0,0.4);border:1px solid rgba(157,78,221,0.2);border-radius:4px;padding:10px 14px;font-family:var(--font-mono);font-size:0.7em;color:var(--purple);word-break:break-all;margin:12px 0}
.proof-badges{display:flex;gap:8px;flex-wrap:wrap}
.pbadge{padding:4px 12px;border-radius:20px;font-family:var(--font-mono);font-size:0.68em;font-weight:600;border:1px solid}
.pb-g{background:rgba(0,255,136,0.08);border-color:rgba(0,255,136,0.3);color:var(--green)}
.pb-p{background:rgba(157,78,221,0.08);border-color:rgba(157,78,221,0.3);color:var(--purple)}
.pb-b{background:rgba(0,229,255,0.08);border-color:rgba(0,229,255,0.3);color:var(--cyan)}
.hist-table{width:100%;border-collapse:collapse;font-size:0.78em}
.hist-table th{text-align:left;padding:8px 10px;font-family:var(--font-mono);font-size:0.65em;color:var(--text3);text-transform:uppercase;letter-spacing:1px;border-bottom:1px solid var(--border);font-weight:400}
.hist-table td{padding:8px 10px;border-bottom:1px solid var(--border);font-family:var(--font-mono);color:var(--text2)}
.hist-table tr:hover td{background:rgba(255,255,255,0.02)}
.t-n{color:var(--green);font-weight:600}.t-m{color:var(--red);font-weight:600}.t-ok{color:var(--green)}.t-na{color:var(--text3)}
.terminal{background:#010407;border:1px solid var(--border);border-radius:6px;padding:12px 14px;font-family:var(--font-mono);font-size:0.72em;max-height:200px;overflow-y:auto;line-height:1.7;position:relative}
.terminal::before{content:'> ZKML TERMINAL';position:absolute;top:8px;right:12px;font-size:0.65em;color:var(--text3)}
.terminal::-webkit-scrollbar{width:3px}.terminal::-webkit-scrollbar-thumb{background:var(--border2)}
.t-line{padding:0}.t-time{color:var(--text3)}.ts{color:var(--green)}.te{color:var(--red)}.ti{color:var(--cyan)}.tw{color:var(--yellow)}.td{color:var(--green)}
.stat-row{display:flex;justify-content:space-between;align-items:center;padding:7px 0;border-bottom:1px solid var(--border)}
.stat-row:last-child{border-bottom:none}.stat-name{font-family:var(--font-mono);font-size:0.72em;color:var(--text3)}.stat-val{font-family:var(--font-mono);font-size:0.72em;color:var(--text);font-weight:600}
.stat-bar{height:3px;background:var(--border);border-radius:2px;margin-top:6px;overflow:hidden}
.stat-bar-fill{height:100%;background:linear-gradient(90deg,var(--cyan),var(--purple));border-radius:2px}
.net-badge{display:flex;align-items:center;gap:7px;padding:7px 14px;border-radius:20px;background:rgba(255,124,0,0.1);border:1px solid rgba(255,124,0,0.3);color:var(--orange);font-family:var(--font-mono);font-size:0.7em;font-weight:600}
.net-dot{width:7px;height:7px;border-radius:50%;background:var(--orange);animation:blink2 2s infinite}
.f5-grid{display:grid;grid-template-columns:repeat(5,1fr);gap:10px;margin-bottom:18px}
.f5-card{background:rgba(0,0,0,0.3);border:1px solid rgba(255,124,0,0.15);border-radius:6px;padding:12px;text-align:center}
.f5-label{font-family:var(--font-mono);font-size:0.62em;color:var(--text3);text-transform:uppercase;margin-bottom:6px;letter-spacing:1px}
.f5-input{width:100%;background:transparent;border:none;border-bottom:1px solid rgba(255,124,0,0.3);color:var(--orange);font-family:var(--font-mono);font-size:0.9em;text-align:center;padding:4px 0;outline:none}
.f5-input:focus{border-bottom-color:var(--orange)}
.tx-card{background:rgba(0,0,0,0.4);border:1px solid rgba(255,124,0,0.3);border-radius:8px;padding:18px;margin-bottom:14px}
.tx-header{display:flex;justify-content:space-between;align-items:center;margin-bottom:14px;flex-wrap:wrap;gap:8px}
.tx-title{font-family:var(--font-mono);font-size:0.65em;color:var(--orange);text-transform:uppercase;letter-spacing:2px}
.tx-confirmed{padding:4px 12px;border-radius:3px;font-family:var(--font-mono);font-size:0.68em;font-weight:700;background:rgba(0,255,136,0.1);border:1px solid rgba(0,255,136,0.4);color:var(--green)}
.tx-grid{display:grid;grid-template-columns:1fr 1fr;gap:10px;margin-bottom:12px}
.tx-field{background:rgba(255,255,255,0.02);border:1px solid var(--border);border-radius:4px;padding:10px 12px}
.tx-field-lbl{font-family:var(--font-mono);font-size:0.62em;color:var(--text3);text-transform:uppercase;letter-spacing:1px;margin-bottom:4px}
.tx-field-val{font-family:var(--font-mono);font-size:0.75em;color:var(--text);word-break:break-all}
.tx-hash-box{background:rgba(0,0,0,0.4);border:1px solid rgba(255,124,0,0.2);border-radius:4px;padding:10px 12px;font-family:var(--font-mono);font-size:0.7em;color:var(--orange);word-break:break-all;margin-bottom:12px}
.oc-pred-banner{border-radius:6px;padding:14px 18px;display:flex;justify-content:space-between;align-items:center;margin-bottom:12px;flex-wrap:wrap;gap:8px;border:1px solid}
.oc-pred-normal{background:linear-gradient(135deg,rgba(0,255,136,0.1),rgba(0,229,255,0.06));border-color:rgba(0,255,136,0.4)}
.oc-pred-malicious{background:linear-gradient(135deg,rgba(255,34,68,0.1),rgba(255,124,0,0.06));border-color:rgba(255,34,68,0.4)}
.oc-pred-text{font-family:var(--font-head);font-size:1em;font-weight:700;letter-spacing:1px}
.contract-panel{background:rgba(0,0,0,0.3);border:1px solid rgba(157,78,221,0.25);border-radius:8px;padding:18px}
.code-scroll{background:#01030a;border:1px solid var(--border);border-radius:4px;padding:12px;max-height:200px;overflow-y:auto;font-family:var(--font-mono);font-size:0.7em;color:var(--text2);line-height:1.7;white-space:pre-wrap;word-break:break-all}
.code-scroll::-webkit-scrollbar{width:3px}.code-scroll::-webkit-scrollbar-thumb{background:var(--border2)}
.remix-steps{list-style:none;counter-reset:step}
.remix-steps li{counter-increment:step;display:flex;align-items:flex-start;gap:10px;padding:8px 0;border-bottom:1px solid var(--border)}
.remix-steps li:last-child{border-bottom:none}
.remix-steps li::before{content:counter(step);min-width:22px;height:22px;border-radius:50%;background:rgba(157,78,221,0.2);border:1px solid rgba(157,78,221,0.4);color:var(--purple);font-size:0.72em;font-weight:700;display:flex;align-items:center;justify-content:center;flex-shrink:0;margin-top:1px}
.remix-step-text{font-family:var(--font-mono);font-size:0.72em;color:var(--text2);line-height:1.6}
.remix-step-text strong{color:var(--text)}
.run-btns{display:flex;gap:10px;justify-content:center;margin-top:14px;padding-top:14px;border-top:1px solid var(--border);flex-wrap:wrap}
.toast{position:fixed;bottom:24px;right:24px;background:var(--card2);border:1px solid var(--border2);border-radius:6px;padding:12px 18px;font-family:var(--font-mono);font-size:0.78em;color:var(--text);z-index:9000;transform:translateX(120%);transition:transform 0.4s cubic-bezier(0.16,1,0.3,1);display:flex;align-items:center;gap:10px;max-width:300px}
.toast.show{transform:translateX(0)}.toast.toast-ok{border-color:rgba(0,255,136,0.4)}.toast.toast-err{border-color:rgba(255,34,68,0.4)}
#results-section,#onchain-results-section{display:none}
</style>
</head>
<body>
<div class="grid-bg"></div>
<div class="glow-orb orb1"></div><div class="glow-orb orb2"></div><div class="glow-orb orb3"></div>
<div class="toast" id="toast"><span class="toast-icon">✅</span><span id="toast-msg">Done</span></div>

<!-- LOGIN -->
<div class="page active" id="login-page">
  <div class="login-container">
    <div class="login-corner tl"></div><div class="login-corner tr"></div>
    <div class="login-corner bl"></div><div class="login-corner br"></div>
    <div class="login-box">
      <div class="login-emblem">
        <div class="login-shield">🛡️</div>
        <div class="login-title">ZKML DEFENSE</div>
        <div class="login-subtitle">Secure Access Portal v2.0</div>
      </div>
      <div class="login-divider"></div>
      <div class="login-status-bar"><div class="status-dot"></div><span id="login-status-text">SYSTEM NOMINAL — AWAITING AUTHENTICATION</span></div>
      <div id="login-form-section">
        <div class="field-group"><div class="field-label">Operator ID</div><div class="field-wrap"><span class="field-icon">◈</span><input class="login-input" type="text" id="login-username" placeholder="enter operator id" autocomplete="off"></div></div>
        <div class="field-group"><div class="field-label">Access Key</div><div class="field-wrap"><span class="field-icon">⬡</span><input class="login-input" type="password" id="login-password" placeholder="enter access key" autocomplete="off"></div><div class="strength-bar" id="strength-bar"><div class="strength-fill" id="strength-fill"></div></div></div>
        <button class="login-btn" id="login-btn">AUTHENTICATE →</button>
        <div class="login-error" id="login-error"><span>⚠</span><span id="login-error-msg">Invalid credentials</span></div>
        <div class="login-toggle">New operator? <a id="go-signup">Register access →</a></div>
      </div>
      <div id="signup-form-section" style="display:none">
        <div class="field-group"><div class="field-label">Choose Operator ID</div><div class="field-wrap"><span class="field-icon">◈</span><input class="login-input" type="text" id="su-username" placeholder="choose operator id" autocomplete="off"></div></div>
        <div class="field-group"><div class="field-label">Choose Access Key</div><div class="field-wrap"><span class="field-icon">⬡</span><input class="login-input" type="password" id="su-password" placeholder="choose access key" autocomplete="off"></div><div class="strength-bar" id="su-strength-bar"><div class="strength-fill" id="su-strength-fill"></div></div></div>
        <div class="field-group"><div class="field-label">Confirm Access Key</div><div class="field-wrap"><span class="field-icon">⬡</span><input class="login-input" type="password" id="su-confirm" placeholder="confirm access key" autocomplete="off"></div></div>
        <button class="login-btn" id="signup-btn">CREATE ACCESS →</button>
        <div class="login-error" id="signup-error"><span>⚠</span><span id="signup-error-msg">Error</span></div>
        <div class="login-toggle">Already registered? <a id="go-login">Sign in →</a></div>
      </div>
      <div class="security-row"><div class="sec-badge"><span></span> AES-256 ENCRYPTED</div><div class="sec-badge"><span></span> ZK SECURED</div><div class="sec-badge"><span></span> CLASSIFIED</div></div>
    </div>
  </div>
</div>

<!-- MODE SELECT -->
<div class="page" id="mode-page">
  <div class="mode-bg"></div>
  <div class="mode-wrap">
    <div class="mode-header">
      <div class="mode-greeting">// SECURE ACCESS GRANTED — SELECT OPERATION MODE</div>
      <div class="mode-title">ZKML <span>Defense</span> IDS</div>
      <div class="mode-subtitle">Choose your verification protocol to proceed</div>
    </div>
    <div class="mode-cards">
      <div class="mode-card offchain" id="choose-offchain">
        <div class="login-corner tl" style="border-color:var(--cyan)"></div><div class="login-corner br" style="border-color:var(--cyan)"></div>
        <div class="mode-card-icon">⚡</div><div class="mode-card-tag">OFF-CHAIN</div><div class="mode-card-title">LOCAL VERIFY</div>
        <p class="mode-card-desc">High-speed ML inference with ZK proof generation off-chain. Full 41-feature model with radar visualization.</p>
        <ul class="mode-card-features"><li>41-feature full neural network</li><li>Instant ZK proof verification</li><li>Live traffic radar chart</li><li>Prediction history log</li><li>System terminal output</li></ul>
        <div class="mode-card-arrow">→</div>
      </div>
      <div class="mode-card onchain" id="choose-onchain">
        <div class="login-corner tl" style="border-color:var(--purple)"></div><div class="login-corner br" style="border-color:var(--purple)"></div>
        <div class="mode-card-icon">⛓️</div><div class="mode-card-tag">ON-CHAIN</div><div class="mode-card-title">BLOCKCHAIN VERIFY</div>
        <p class="mode-card-desc">Deploy verification to Ethereum Sepolia. Compact 5-feature model with ZK proof validated by smart contract.</p>
        <ul class="mode-card-features"><li>5-feature compact model</li><li>Ethereum Sepolia simulation</li><li>Solidity verifier contract</li><li>Transaction receipt + calldata</li><li>Remix IDE deployment guide</li></ul>
        <div class="mode-card-arrow">→</div>
      </div>
    </div>
    <div class="mode-user-info">Signed in as <span id="mode-username-display">OPERATOR</span> &nbsp;·&nbsp; <a style="color:var(--text3);cursor:pointer;font-family:var(--font-mono)" id="mode-logout">Logout</a></div>
  </div>
</div>

<!-- OFF-CHAIN DASHBOARD -->
<div class="page" id="offchain-page">
  <div class="dash-wrap">
    <nav class="topnav">
      <div class="nav-left"><div class="nav-logo"><div class="nav-logo-icon">🛡️</div><div class="nav-logo-text">ZKML DEFENSE IDS</div></div><div class="nav-mode-badge off">⚡ OFF-CHAIN</div></div>
      <div class="nav-stats"><div class="nav-stat"><div class="nav-stat-val">125,973</div><div class="nav-stat-lbl">Training Samples</div></div><div class="nav-stat"><div class="nav-stat-val">78.66%</div><div class="nav-stat-lbl">Accuracy</div></div><div class="nav-stat"><div class="nav-stat-val">41 / 5</div><div class="nav-stat-lbl">Features</div></div></div>
      <div class="nav-right"><div class="nav-pill g"><div class="nav-pill-dot"></div>Model Online</div><div class="nav-pill b"><div class="nav-pill-dot"></div>ZK Ready</div><button class="nav-btn" id="off-switch-mode">⇄ Switch Mode</button><button class="nav-btn" id="off-logout">⏻ Logout</button></div>
    </nav>
    <div class="dash-content">
      <div class="pipeline-card">
        <div class="pipe-step" id="off-pipe1"><div class="pipe-icon">🧠</div><div class="pipe-lbl">ML Inference</div><div class="pipe-status" id="off-ps1">Standby</div></div>
        <div class="pipe-arrow">→</div>
        <div class="pipe-step" id="off-pipe2"><div class="pipe-icon">🔏</div><div class="pipe-lbl">ZK Proof</div><div class="pipe-status" id="off-ps2">Standby</div></div>
        <div class="pipe-arrow">→</div>
        <div class="pipe-step" id="off-pipe3"><div class="pipe-icon">✅</div><div class="pipe-lbl">Verify</div><div class="pipe-status" id="off-ps3">Standby</div></div>
      </div>
      <div class="grid-2">
        <div class="card">
          <div class="card-title">Network Traffic Features (41)</div>
          <div class="btn-controls"><button class="btn btn-green" id="off-btn-normal">✅ Normal Traffic</button><button class="btn btn-red" id="off-btn-malicious">🚨 Malicious Traffic</button><button class="btn btn-ghost" id="off-btn-clear">Clear</button></div>
          <div class="feature-scroll"><div class="feature-grid" id="off-inputs"></div></div>
          <div class="run-btns"><button class="btn btn-purple" id="off-btn-pipeline">🚀 Run Full Pipeline</button><button class="btn btn-cyan" id="off-btn-predict">⚡ Off-Chain Verify</button></div>
          <div class="loading-wrap" id="off-loading"><div class="loading-text" id="off-loading-text">⚡ Processing...</div><div class="prog-bar"><div class="prog-fill"></div></div></div>
        </div>
        <div style="display:flex;flex-direction:column;gap:16px">
          <div class="card" style="flex:1"><div class="card-title">Traffic Profile Radar</div><canvas id="off-radarChart" height="200"></canvas></div>
          <div class="card">
            <div class="card-title">Model Info</div>
            <div class="stat-row"><span class="stat-name">Full Model</span><span class="stat-val">MLP [41→64→32→1]</span></div>
            <div class="stat-row"><span class="stat-name">Small Model</span><span class="stat-val">MLP [5→10→1]</span></div>
            <div class="stat-row"><span class="stat-name">Dataset</span><span class="stat-val">NSL-KDD</span></div>
            <div class="stat-row"><span class="stat-name">ZK Backend</span><span class="stat-val">EZKL v23</span></div>
            <div class="stat-row"><span class="stat-name">Proof System</span><span class="stat-val">KZG/Groth16</span></div>
            <div style="margin-top:8px"><div style="display:flex;justify-content:space-between;font-family:var(--font-mono);font-size:0.72em;margin-bottom:4px"><span style="color:var(--text3)">Full Model Accuracy</span><span style="color:var(--cyan)">78.66%</span></div><div class="stat-bar"><div class="stat-bar-fill" style="width:78.66%"></div></div></div>
          </div>
        </div>
      </div>
      <div id="results-section">
        <div id="off-result-content"></div>
        <div class="grid-2">
          <div class="card"><div class="card-title">Prediction History <span id="off-history-count" style="margin-left:8px;font-size:0.85em;color:var(--text3)">0 records</span></div><div style="overflow-x:auto"><table class="hist-table"><thead><tr><th>#</th><th>Time</th><th>Prediction</th><th>Confidence</th><th>Proof</th></tr></thead><tbody id="off-history-body"></tbody></table></div></div>
          <div class="card"><div class="card-title">System Terminal</div><div class="terminal" id="off-terminal"><div class="t-line"><span class="t-time">[BOOT] </span><span class="ts">ZKML Defense IDS v2.0 — Off-Chain Mode</span></div><div class="t-line"><span class="t-time">[INFO] </span><span class="ti">Full model (41-feat) loaded</span></div><div class="t-line"><span class="t-time">[OK]   </span><span class="ts">System ready. Awaiting input...</span></div></div></div>
        </div>
      </div>
    </div>
  </div>
</div>

<!-- ON-CHAIN DASHBOARD -->
<div class="page" id="onchain-page">
  <div class="dash-wrap">
    <nav class="topnav">
      <div class="nav-left"><div class="nav-logo"><div class="nav-logo-icon">🛡️</div><div class="nav-logo-text">ZKML DEFENSE IDS</div></div><div class="nav-mode-badge on">⛓️ ON-CHAIN</div></div>
      <div class="nav-stats"><div class="nav-stat"><div class="nav-stat-val">125,973</div><div class="nav-stat-lbl">Training Samples</div></div><div class="nav-stat"><div class="nav-stat-val">78.66%</div><div class="nav-stat-lbl">Accuracy</div></div><div class="nav-stat"><div class="nav-stat-val">5 feat</div><div class="nav-stat-lbl">Features</div></div></div>
      <div class="nav-right"><div class="nav-pill g"><div class="nav-pill-dot"></div>Model Online</div><div class="nav-pill p"><div class="nav-pill-dot"></div>On-Chain Ready</div><button class="nav-btn" id="on-switch-mode">⇄ Switch Mode</button><button class="nav-btn" id="on-logout">⏻ Logout</button></div>
    </nav>
    <div class="dash-content">
      <div class="pipeline-card">
        <div class="pipe-step" id="on-pipe1"><div class="pipe-icon">🧠</div><div class="pipe-lbl">ML Inference</div><div class="pipe-status" id="on-ps1">Standby</div></div>
        <div class="pipe-arrow">→</div>
        <div class="pipe-step" id="on-pipe2"><div class="pipe-icon">🔏</div><div class="pipe-lbl">ZK Proof</div><div class="pipe-status" id="on-ps2">Standby</div></div>
        <div class="pipe-arrow">→</div>
        <div class="pipe-step" id="on-pipe3"><div class="pipe-icon">⛓️</div><div class="pipe-lbl">Blockchain</div><div class="pipe-status" id="on-ps3">Standby</div></div>
      </div>
      <div class="grid-2">
        <div class="card">
          <div class="card-title orange">Blockchain Verification (5 Features)</div>
          <div class="net-badge" style="margin-bottom:14px;display:inline-flex"><div class="net-dot"></div>Sepolia Testnet</div>
          <div class="btn-controls"><button class="btn btn-green" id="oc-preset-normal">✅ Normal Preset</button><button class="btn btn-red" id="oc-preset-malicious">🚨 Malicious Preset</button></div>
          <div class="f5-grid">
            <div class="f5-card"><div class="f5-label">src_bytes</div><input class="f5-input" id="oc0" type="number" value="181" step="any"></div>
            <div class="f5-card"><div class="f5-label">dst_bytes</div><input class="f5-input" id="oc1" type="number" value="5450" step="any"></div>
            <div class="f5-card"><div class="f5-label">count</div><input class="f5-input" id="oc2" type="number" value="8" step="any"></div>
            <div class="f5-card"><div class="f5-label">serror_rate</div><input class="f5-input" id="oc3" type="number" value="0" step="0.01"></div>
            <div class="f5-card"><div class="f5-label">rerror_rate</div><input class="f5-input" id="oc4" type="number" value="0" step="0.01"></div>
          </div>
          <div class="run-btns"><button class="btn btn-orange" id="btn-onchain">⛓️ Simulate On-Chain Verify</button><a class="btn btn-ghost" href="https://remix.ethereum.org" target="_blank">🔗 Remix IDE</a></div>
          <div class="loading-wrap" id="oc-loading"><div class="loading-text" id="oc-loading-text">⛓️ Submitting...</div><div class="prog-bar"><div class="prog-fill" style="background:linear-gradient(90deg,var(--orange),var(--purple))"></div></div></div>
        </div>
        <div style="display:flex;flex-direction:column;gap:16px">
          <div class="card" style="flex:1"><div class="card-title orange">Traffic Profile Radar</div><canvas id="on-radarChart" height="200"></canvas></div>
          <div class="card">
            <div class="card-title">Model Info</div>
            <div class="stat-row"><span class="stat-name">Small Model</span><span class="stat-val">MLP [5→10→1]</span></div>
            <div class="stat-row"><span class="stat-name">Network</span><span class="stat-val" style="color:var(--orange)">Sepolia Testnet</span></div>
            <div class="stat-row"><span class="stat-name">Proof System</span><span class="stat-val">KZG/Groth16</span></div>
            <div class="stat-row"><span class="stat-name">ZK Backend</span><span class="stat-val">EZKL v23</span></div>
          </div>
        </div>
      </div>
      <div id="onchain-results-section">
        <div id="oc-result-content"></div>
        <div id="oc-tx-section" style="display:none">
          <div class="tx-card">
            <div class="tx-header"><div class="tx-title">📦 Transaction Receipt</div><span class="tx-confirmed">✓ CONFIRMED</span></div>
            <div class="oc-pred-banner" id="oc-pred-banner2"><div class="oc-pred-text" id="oc-pred-text2"></div><div style="font-family:var(--font-mono);font-size:0.8em" id="oc-conf-text2"></div></div>
            <div style="font-family:var(--font-mono);font-size:0.65em;color:var(--text3);margin-bottom:6px;text-transform:uppercase;letter-spacing:1px">Transaction Hash:</div>
            <div class="tx-hash-box" id="oc-tx-hash2"></div>
            <div class="tx-grid">
              <div class="tx-field"><div class="tx-field-lbl">Block Number</div><div class="tx-field-val" id="oc-block2"></div></div>
              <div class="tx-field"><div class="tx-field-lbl">Gas Used</div><div class="tx-field-val" id="oc-gas2"></div></div>
              <div class="tx-field"><div class="tx-field-lbl">Network</div><div class="tx-field-val" id="oc-network2"></div></div>
              <div class="tx-field"><div class="tx-field-lbl">ZK Proof Valid</div><div class="tx-field-val" id="oc-proof-valid2"></div></div>
              <div class="tx-field"><div class="tx-field-lbl">Contract Address</div><div class="tx-field-val" id="oc-contract2" style="font-size:0.7em"></div></div>
              <div class="tx-field"><div class="tx-field-lbl">Calldata Preview</div><div class="tx-field-val" id="oc-calldata2" style="font-size:0.65em;color:var(--purple)"></div></div>
            </div>
          </div>
          <div class="grid-2">
            <div class="contract-panel"><div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:12px"><div class="card-title purple" style="margin-bottom:0">📄 Verifier_5.sol</div><button class="btn btn-ghost" style="padding:5px 10px;font-size:0.7em" onclick="copySol()">📋 Copy</button></div><div class="code-scroll" id="sol-code"></div></div>
            <div class="contract-panel">
              <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:12px"><div class="card-title purple" style="margin-bottom:0">📡 Calldata</div><button class="btn btn-ghost" style="padding:5px 10px;font-size:0.7em" onclick="copyCalldata()">📋 Copy</button></div>
              <div class="code-scroll" id="calldata-display"></div>
              <div style="margin-top:14px">
                <div class="card-title" style="margin-bottom:8px">How to verify on Remix:</div>
                <ol class="remix-steps">
                  <li><div class="remix-step-text">Go to <strong>remix.ethereum.org</strong> → New File → paste <strong>Verifier_5.sol</strong></div></li>
                  <li><div class="remix-step-text">Compile with <strong>Solidity 0.8.20</strong>, optimizer enabled</div></li>
                  <li><div class="remix-step-text">Deploy to <strong>Injected Provider (MetaMask)</strong> on Sepolia</div></li>
                  <li><div class="remix-step-text">Call <strong>verifyProof()</strong> and paste the calldata above</div></li>
                  <li><div class="remix-step-text">Result <strong>true</strong> = ZK proof verified on-chain ✅</div></li>
                </ol>
              </div>
            </div>
          </div>
        </div>
        <div class="grid-2" style="margin-top:16px">
          <div class="card"><div class="card-title orange">Prediction History <span id="oc-history-count" style="margin-left:8px;font-size:0.85em;color:var(--text3)">0 records</span></div><div style="overflow-x:auto"><table class="hist-table"><thead><tr><th>#</th><th>Time</th><th>Prediction</th><th>Confidence</th><th>Block</th></tr></thead><tbody id="oc-history-body"></tbody></table></div></div>
          <div class="card"><div class="card-title orange">System Terminal</div><div class="terminal" id="oc-terminal"><div class="t-line"><span class="t-time">[BOOT] </span><span class="ts">ZKML Defense IDS v2.0 — On-Chain Mode</span></div><div class="t-line"><span class="t-time">[INFO] </span><span class="ti">Small model (5-feat) loaded</span></div><div class="t-line"><span class="t-time">[OK]   </span><span class="ts">Ready for on-chain verification...</span></div></div></div>
        </div>
      </div>
    </div>
  </div>
</div>

<script>
var currentUser=null,users=JSON.parse(localStorage.getItem('zkml_users')||'{}');
var offHistoryData=[],onHistoryData=[],offRadarChart=null,onRadarChart=null,fullSolCode='',fullCalldata='';
var fnames=["duration","protocol_type","service","flag","src_bytes","dst_bytes","land","wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised","root_shell","su_attempted","num_root","num_file_creations","num_shells","num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count","srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate"];
var normalPreset=[0,1,49,10,181,5450,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,8,8,0,0,0,0,1,0,0,9,9,1,0,0.11,0,0,0,0,0];
var maliciousPreset=[0,2,7,7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,229,10,0,0,1,1,0.04,0.06,0,255,10,0.04,0.06,0,0,0,0,1,1];

function showPage(id){document.querySelectorAll('.page').forEach(function(p){p.classList.remove('active','slide-in');});var pg=document.getElementById(id);pg.classList.add('active','slide-in');setTimeout(function(){pg.classList.remove('slide-in');},700);}
function showToast(msg,type){var t=document.getElementById('toast');document.getElementById('toast-msg').textContent=msg;t.className='toast show '+(type==='err'?'toast-err':'toast-ok');document.querySelector('#toast .toast-icon').textContent=type==='err'?'⚠️':'✅';setTimeout(function(){t.classList.remove('show');},3000);}

var msgs=['SYSTEM NOMINAL — AWAITING AUTHENTICATION','ZK PROOF ENGINE ONLINE','NSL-KDD MODEL LOADED','SEPOLIA TESTNET CONNECTED'],mIdx=0;
setInterval(function(){mIdx=(mIdx+1)%msgs.length;document.getElementById('login-status-text').textContent=msgs[mIdx];},3000);

document.getElementById('go-signup').onclick=function(){document.getElementById('login-form-section').style.display='none';document.getElementById('signup-form-section').style.display='block';};
document.getElementById('go-login').onclick=function(){document.getElementById('signup-form-section').style.display='none';document.getElementById('login-form-section').style.display='block';};

function pwStrength(pw){var s=0;if(pw.length>=4)s+=25;if(pw.length>=8)s+=25;if(/[A-Z]/.test(pw))s+=25;if(/[0-9!@#$%]/.test(pw))s+=25;return s;}
document.getElementById('login-password').oninput=function(){var s=pwStrength(this.value);document.getElementById('strength-bar').style.display='block';var f=document.getElementById('strength-fill');f.style.width=s+'%';f.style.background=s<=25?'#ff2244':s<=50?'#ffd700':s<=75?'#00b4cc':'#00ff88';};
document.getElementById('su-password').oninput=function(){var s=pwStrength(this.value);document.getElementById('su-strength-bar').style.display='block';var f=document.getElementById('su-strength-fill');f.style.width=s+'%';f.style.background=s<=25?'#ff2244':s<=50?'#ffd700':s<=75?'#00b4cc':'#00ff88';};

document.getElementById('login-btn').onclick=function(){var u=document.getElementById('login-username').value.trim(),p=document.getElementById('login-password').value;if(!u||!p){showE('login-error','login-error-msg','All fields required');return;}if(users[u]&&users[u]===btoa(p)){currentUser=u;document.getElementById('mode-username-display').textContent=u.toUpperCase();showPage('mode-page');}else if(!users[u]){showE('login-error','login-error-msg','Operator ID not found.');}else{showE('login-error','login-error-msg','Invalid credentials.');}};
document.getElementById('signup-btn').onclick=function(){var u=document.getElementById('su-username').value.trim(),p=document.getElementById('su-password').value,c=document.getElementById('su-confirm').value;if(!u||!p||!c){showE('signup-error','signup-error-msg','All fields required');return;}if(p!==c){showE('signup-error','signup-error-msg','Keys do not match');return;}if(p.length<4){showE('signup-error','signup-error-msg','Min 4 chars');return;}if(users[u]){showE('signup-error','signup-error-msg','ID already exists');return;}users[u]=btoa(p);localStorage.setItem('zkml_users',JSON.stringify(users));currentUser=u;document.getElementById('mode-username-display').textContent=u.toUpperCase();showPage('mode-page');showToast('Welcome, '+u.toUpperCase()+'!');};
function showE(eid,mid,msg){var e=document.getElementById(eid);document.getElementById(mid).textContent=msg;e.style.display='flex';}
document.getElementById('login-password').onkeydown=function(e){if(e.key==='Enter')document.getElementById('login-btn').click();};
document.getElementById('su-confirm').onkeydown=function(e){if(e.key==='Enter')document.getElementById('signup-btn').click();};

document.getElementById('choose-offchain').onclick=function(){showPage('offchain-page');initOffChain();};
document.getElementById('choose-onchain').onclick=function(){showPage('onchain-page');initOnChain();};
document.getElementById('mode-logout').onclick=logout;
document.getElementById('off-switch-mode').onclick=function(){showPage('mode-page');};
document.getElementById('on-switch-mode').onclick=function(){showPage('mode-page');};
document.getElementById('off-logout').onclick=logout;
document.getElementById('on-logout').onclick=logout;
function logout(){currentUser=null;showPage('login-page');showToast('Signed out');}

var offInited=false;
function initOffChain(){if(offInited)return;offInited=true;var g=document.getElementById('off-inputs');for(var i=0;i<fnames.length;i++){var d=document.createElement('div');d.className='input-group';d.innerHTML='<label>'+fnames[i]+'</label><input type="number" id="off-f'+i+'" value="0" step="any">';g.appendChild(d);}updateOffRadar(new Array(41).fill(0));
document.getElementById('off-btn-normal').onclick=function(){for(var i=0;i<normalPreset.length;i++)document.getElementById('off-f'+i).value=normalPreset[i];updateOffRadar(normalPreset);addLog('off','Normal traffic preset loaded','i');};
document.getElementById('off-btn-malicious').onclick=function(){for(var i=0;i<maliciousPreset.length;i++)document.getElementById('off-f'+i).value=maliciousPreset[i];updateOffRadar(maliciousPreset);addLog('off','Malicious traffic preset loaded','w');};
document.getElementById('off-btn-clear').onclick=function(){for(var i=0;i<fnames.length;i++)document.getElementById('off-f'+i).value=0;addLog('off','Inputs cleared','i');};
document.getElementById('off-btn-predict').onclick=runOffPredict;
document.getElementById('off-btn-pipeline').onclick=runOffPipeline;}

var onInited=false;
function initOnChain(){if(onInited)return;onInited=true;updateOnRadar([181,5450,8,0,0]);
document.getElementById('oc-preset-normal').onclick=function(){document.getElementById('oc0').value=181;document.getElementById('oc1').value=5450;document.getElementById('oc2').value=8;document.getElementById('oc3').value=0;document.getElementById('oc4').value=0;updateOnRadar([181,5450,8,0,0]);addLog('oc','Normal preset','i');};
document.getElementById('oc-preset-malicious').onclick=function(){document.getElementById('oc0').value=0;document.getElementById('oc1').value=0;document.getElementById('oc2').value=229;document.getElementById('oc3').value=0;document.getElementById('oc4').value=1;updateOnRadar([0,0,229,0,1]);addLog('oc','Malicious preset','w');};
document.getElementById('btn-onchain').onclick=runOnChain;}

function getOffF(){var f=[];for(var i=0;i<fnames.length;i++)f.push(parseFloat(document.getElementById('off-f'+i).value)||0);return f;}
function getF5(){return[parseFloat(document.getElementById('oc0').value)||0,parseFloat(document.getElementById('oc1').value)||0,parseFloat(document.getElementById('oc2').value)||0,parseFloat(document.getElementById('oc3').value)||0,parseFloat(document.getElementById('oc4').value)||0];}

function setPipe(pfx,n,status,cls){var p=document.getElementById(pfx+'-pipe'+n),s=document.getElementById(pfx+'-ps'+n);if(!p)return;p.className='pipe-step'+(cls?' '+cls:'');s.textContent=status;s.style.color=cls==='active'?'#00ff88':cls==='processing'?'#ffd700':cls==='error'?'#ff2244':'#3a5a78';}
function addLog(pfx,msg,type){var tid=pfx==='oc'?'oc-terminal':'off-terminal';var el=document.getElementById(tid);var now=new Date();var time=now.getHours().toString().padStart(2,'0')+':'+now.getMinutes().toString().padStart(2,'0')+':'+now.getSeconds().toString().padStart(2,'0');var cls=type==='s'?'ts':type==='e'?'te':type==='i'?'ti':type==='w'?'tw':'td';el.innerHTML+='<div class="t-line"><span class="t-time">['+time+'] </span><span class="'+cls+'">'+msg+'</span></div>';el.scrollTop=el.scrollHeight;}
function getAttackType(f,m){if(!m)return{label:'NORMAL TRAFFIC',cls:'at-normal',icon:'✅'};var se=f[24]||0,cnt=f[22]||0,db=f[5]||0;if(se>0.5)return{label:'DoS ATTACK',cls:'at-dos',icon:'⚡'};if(cnt>100)return{label:'PROBE ATTACK',cls:'at-probe',icon:'🔍'};if(db>1000)return{label:'R2L ATTACK',cls:'at-r2l',icon:'🌐'};return{label:'U2R ATTACK',cls:'at-r2l',icon:'💀'};}

function updateOffRadar(f){var labels=['Duration','Src Bytes','Dst Bytes','Count','Srv Count','Dst Host','Serror Rate','Rerror Rate'],idx=[0,4,5,22,23,31,24,26],mx=[10000,100000,100000,512,512,255,1,1];var vals=idx.map(function(id,i){return Math.min((f[id]||0)/mx[i],1);});if(offRadarChart)offRadarChart.destroy();var ctx=document.getElementById('off-radarChart').getContext('2d');offRadarChart=new Chart(ctx,{type:'radar',data:{labels:labels,datasets:[{label:'Traffic',data:vals,backgroundColor:'rgba(0,229,255,0.08)',borderColor:'rgba(0,229,255,0.6)',pointBackgroundColor:'#00e5ff',pointBorderColor:'#1a2d4a',pointRadius:4}]},options:{responsive:true,plugins:{legend:{display:false}},scales:{r:{angleLines:{color:'rgba(255,255,255,0.04)'},grid:{color:'rgba(255,255,255,0.04)'},pointLabels:{color:'#3a5a78',font:{family:'Share Tech Mono',size:10}},ticks:{display:false},min:0,max:1}}}});}
function updateOnRadar(f5){var labels=['Src Bytes','Dst Bytes','Count','Serror Rate','Rerror Rate'],mx=[100000,100000,512,1,1];var vals=f5.map(function(v,i){return Math.min((v||0)/mx[i],1);});if(onRadarChart)onRadarChart.destroy();var ctx=document.getElementById('on-radarChart').getContext('2d');onRadarChart=new Chart(ctx,{type:'radar',data:{labels:labels,datasets:[{label:'Traffic',data:vals,backgroundColor:'rgba(255,124,0,0.08)',borderColor:'rgba(255,124,0,0.6)',pointBackgroundColor:'#ff7c00',pointBorderColor:'#1a2d4a',pointRadius:4}]},options:{responsive:true,plugins:{legend:{display:false}},scales:{r:{angleLines:{color:'rgba(255,255,255,0.04)'},grid:{color:'rgba(255,255,255,0.04)'},pointLabels:{color:'#3a5a78',font:{family:'Share Tech Mono',size:10}},ticks:{display:false},min:0,max:1}}}});}

function addOffHistory(pred,conf,vf){offHistoryData.unshift({pred:pred,conf:conf,vf:vf,time:new Date().toLocaleTimeString()});var m=pred.indexOf('MALICIOUS')!==-1;document.getElementById('off-history-body').innerHTML='<tr><td>'+offHistoryData.length+'</td><td>'+offHistoryData[0].time+'</td><td class="'+(m?'t-m':'t-n')+'">'+(m?'🚨 MALICIOUS':'✅ NORMAL')+'</td><td>'+conf+'%</td><td class="'+(vf===null?'t-na':vf?'t-ok':'te')+'">'+(vf===null?'—':vf?'✓':'✗')+'</td></tr>'+document.getElementById('off-history-body').innerHTML;document.getElementById('off-history-count').textContent=offHistoryData.length+' records';}
function addOnHistory(pred,conf,block){onHistoryData.unshift({pred:pred,conf:conf,block:block,time:new Date().toLocaleTimeString()});var m=pred.indexOf('MALICIOUS')!==-1;document.getElementById('oc-history-body').innerHTML='<tr><td>'+onHistoryData.length+'</td><td>'+onHistoryData[0].time+'</td><td class="'+(m?'t-m':'t-n')+'">'+(m?'🚨 MALICIOUS':'✅ NORMAL')+'</td><td>'+conf+'%</td><td style="color:var(--orange)">#'+block+'</td></tr>'+document.getElementById('oc-history-body').innerHTML;document.getElementById('oc-history-count').textContent=onHistoryData.length+' records';}

function showOffResult(pd,vd){var m=pd.prediction.indexOf('MALICIOUS')!==-1,f=getOffF(),at=getAttackType(f,m),cf=pd.confidence;var h='<div class="result-card '+(m?'result-malicious':'result-normal')+'"><div class="result-header"><div><div class="result-title '+(m?'m':'n')+'">'+(m?'🚨 THREAT DETECTED':'✅ TRAFFIC NORMAL')+'</div><div class="attack-tag '+at.cls+'" style="margin-top:8px">'+at.icon+' '+at.label+'</div></div><span class="threat-badge '+(m?(cf>90?'threat-critical':'threat-high'):'threat-low')+'">'+(m?(cf>90?'CRITICAL':'HIGH RISK'):'LOW RISK')+'</span></div><div class="metrics-row"><div class="metric"><div class="metric-val" style="color:'+(m?'#ff2244':'#00ff88')+'">'+(m?'BLOCK':'ALLOW')+'</div><div class="metric-lbl">Decision</div></div><div class="metric"><div class="metric-val">'+cf+'%</div><div class="metric-lbl">Confidence</div></div><div class="metric"><div class="metric-val">'+pd.raw_score+'</div><div class="metric-lbl">Risk Score</div></div><div class="metric"><div class="metric-val" style="color:'+(m?'#ff2244':'#00ff88')+'">'+(m?(cf>90?'CRITICAL':'HIGH'):'LOW')+'</div><div class="metric-lbl">Threat Level</div></div></div><div><div style="display:flex;justify-content:space-between;font-family:var(--font-mono);font-size:0.72em;margin-bottom:6px"><span style="color:var(--text3)">Confidence</span><span style="color:'+(m?'var(--red)':'var(--green)')+'">'+cf+'%</span></div><div class="conf-bar"><div class="conf-fill '+(m?'fill-m':'fill-n')+'" style="width:'+cf+'%"></div></div></div></div>';
if(vd){h+='<div class="proof-panel"><div class="proof-header"><div class="proof-label">🔐 Zero-Knowledge Proof</div><span class="proof-verified '+(vd.verified?'pv-ok':'pv-fail')+'">'+(vd.verified?'✓ VERIFIED':'✗ FAILED')+'</span></div><div style="display:grid;grid-template-columns:repeat(3,1fr);gap:10px;margin-bottom:12px"><div class="metric"><div class="metric-val" style="color:'+(vd.verified?'#00ff88':'#ff2244')+';font-size:0.9em">'+(vd.verified?'TRUE':'FALSE')+'</div><div class="metric-lbl">Proof Valid</div></div><div class="metric"><div class="metric-val" style="font-size:0.85em">'+(vd.gas_used||'N/A')+'</div><div class="metric-lbl">Gas Est.</div></div><div class="metric"><div class="metric-val" style="color:var(--purple);font-size:0.85em">KZG</div><div class="metric-lbl">System</div></div></div><div class="proof-hash-box">PROOF HASH: '+(vd.proof_hash||'N/A')+'</div><div class="proof-badges"><span class="pbadge pb-g">✅ Model Integrity</span><span class="pbadge pb-p">🔒 Input Private</span><span class="pbadge pb-b">⛓️ Cryptographic Proof</span></div></div>';}
document.getElementById('off-result-content').innerHTML=h;document.getElementById('results-section').style.display='block';}

function runOffPredict(){var f=getOffF();document.getElementById('off-loading').style.display='block';document.getElementById('results-section').style.display='none';document.getElementById('off-loading-text').textContent='⚡ Running ML inference...';setPipe('off',1,'Processing...','processing');addLog('off','ML inference started','i');updateOffRadar(f);
fetch('/predict',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({features:f})}).then(function(r){return r.json();}).then(function(d){document.getElementById('off-loading').style.display='none';if(d.error){setPipe('off',1,'Error','error');addLog('off',d.error,'e');return;}setPipe('off',1,'Complete','active');addLog('off','Prediction: '+d.prediction+' ('+d.confidence+'%)','s');addOffHistory(d.prediction,d.confidence,null);showOffResult(d,null);showToast(d.prediction+' — '+d.confidence+'% confidence');}).catch(function(e){document.getElementById('off-loading').style.display='none';addLog('off',''+e,'e');});}

function runOffPipeline(){var f=getOffF();document.getElementById('off-loading').style.display='block';document.getElementById('results-section').style.display='none';setPipe('off',1,'Processing...','processing');setPipe('off',2,'Standby','');setPipe('off',3,'Standby','');addLog('off','Full pipeline initiated','i');updateOffRadar(f);document.getElementById('off-loading-text').textContent='🧠 Running ML inference...';
fetch('/predict',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({features:f})}).then(function(r){return r.json();}).then(function(pd){if(pd.error){document.getElementById('off-loading').style.display='none';setPipe('off',1,'Error','error');addLog('off',pd.error,'e');return;}setPipe('off',1,'Complete','active');setPipe('off',2,'Verifying...','processing');addLog('off','ML done: '+pd.prediction,'s');document.getElementById('off-loading-text').textContent='🔏 Verifying ZK proof...';addLog('off','Verifying ZK proof...','i');
return fetch('/verify_proof',{method:'POST',headers:{'Content-Type':'application/json'}}).then(function(r){return r.json();}).then(function(vd){document.getElementById('off-loading').style.display='none';if(vd.error){setPipe('off',2,'Error','error');setPipe('off',3,'Error','error');addLog('off','Proof: '+vd.error,'e');}else{setPipe('off',2,'Complete','active');setPipe('off',3,vd.verified?'Verified!':'Failed',vd.verified?'active':'error');addLog('off','ZK: '+(vd.verified?'VERIFIED':'FAILED'),vd.verified?'s':'e');}addOffHistory(pd.prediction,pd.confidence,vd.verified);showOffResult(pd,vd);showToast(pd.prediction+' — '+(vd.verified?'Proof verified':'Proof failed'));});}).catch(function(e){document.getElementById('off-loading').style.display='none';addLog('off',''+e,'e');});}

function runOnChain(){var f5=getF5();document.getElementById('oc-loading').style.display='block';document.getElementById('oc-tx-section').style.display='none';document.getElementById('onchain-results-section').style.display='block';setPipe('on',1,'Processing...','processing');setPipe('on',2,'Standby','');setPipe('on',3,'Standby','');addLog('oc','On-chain verification initiated','i');updateOnRadar(f5);
var steps=['Broadcasting transaction...','Confirming block...','Verifying ZK proof...'],si=0;document.getElementById('oc-loading-text').textContent=steps[0];var iv=setInterval(function(){si++;if(si<steps.length)document.getElementById('oc-loading-text').textContent=steps[si];},400);
setTimeout(function(){clearInterval(iv);fetch('/onchain_demo',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({features_5:f5})}).then(function(r){return r.json();}).then(function(d){document.getElementById('oc-loading').style.display='none';if(d.error){addLog('oc','Error: '+d.error,'e');return;}setPipe('on',1,'Complete','active');setPipe('on',2,'Complete','active');setPipe('on',3,d.proof_valid?'Verified!':'Simulated',d.proof_valid?'active':'processing');
var m=d.prediction.indexOf('MALICIOUS')!==-1;var b=document.getElementById('oc-pred-banner2');b.className='oc-pred-banner '+(m?'oc-pred-malicious':'oc-pred-normal');document.getElementById('oc-pred-text2').style.color=m?'#ff2244':'#00ff88';document.getElementById('oc-pred-text2').textContent=(m?'🚨 MALICIOUS TRAFFIC':'✅ NORMAL TRAFFIC');document.getElementById('oc-conf-text2').textContent='Confidence: '+d.confidence+'%  |  Score: '+d.raw_score;document.getElementById('oc-tx-hash2').textContent=d.tx_hash;document.getElementById('oc-block2').textContent='#'+d.block_number.toLocaleString();document.getElementById('oc-gas2').textContent=d.gas_used.toLocaleString()+' gas';document.getElementById('oc-network2').textContent=d.network;document.getElementById('oc-proof-valid2').textContent=d.proof_valid?'✅ TRUE':'⚠️ Simulated';document.getElementById('oc-proof-valid2').style.color=d.proof_valid?'#00ff88':'#ffd700';document.getElementById('oc-contract2').textContent=d.contract_address;document.getElementById('oc-calldata2').textContent=d.calldata_preview;
addLog('oc','TX: '+d.tx_hash.substring(0,20)+'...','s');addLog('oc','Block #'+d.block_number+' | Gas: '+d.gas_used.toLocaleString(),'i');addOnHistory(d.prediction,d.confidence,d.block_number.toLocaleString());
var h='<div class="result-card '+(m?'result-malicious':'result-normal')+'" style="margin-bottom:16px"><div class="result-header"><div class="result-title '+(m?'m':'n')+'">'+(m?'🚨 THREAT DETECTED':'✅ TRAFFIC NORMAL')+'</div><span class="threat-badge '+(m?(d.confidence>90?'threat-critical':'threat-high'):'threat-low')+'">'+(m?(d.confidence>90?'CRITICAL':'HIGH RISK'):'LOW RISK')+'</span></div><div class="metrics-row"><div class="metric"><div class="metric-val" style="color:'+(m?'#ff2244':'#00ff88')+'">'+(m?'BLOCK':'ALLOW')+'</div><div class="metric-lbl">Decision</div></div><div class="metric"><div class="metric-val">'+d.confidence+'%</div><div class="metric-lbl">Confidence</div></div><div class="metric"><div class="metric-val">'+d.raw_score+'</div><div class="metric-lbl">Risk Score</div></div><div class="metric"><div class="metric-val" style="color:var(--orange)">ON-CHAIN</div><div class="metric-lbl">Mode</div></div></div><div><div style="display:flex;justify-content:space-between;font-family:var(--font-mono);font-size:0.72em;margin-bottom:6px"><span style="color:var(--text3)">Confidence</span><span>'+d.confidence+'%</span></div><div class="conf-bar"><div class="conf-fill '+(m?'fill-m':'fill-n')+'" style="width:'+d.confidence+'%"></div></div></div></div>';
document.getElementById('oc-result-content').innerHTML=h;
fetch('/get_contract').then(function(r){return r.json();}).then(function(c){fullSolCode=c.sol_code;document.getElementById('sol-code').textContent=c.sol_code;});
fetch('/get_calldata').then(function(r){return r.json();}).then(function(c){fullCalldata=c.calldata;document.getElementById('calldata-display').textContent=c.calldata;});
document.getElementById('oc-tx-section').style.display='block';showToast(d.prediction+' — Block #'+d.block_number);}).catch(function(e){clearInterval(iv);document.getElementById('oc-loading').style.display='none';addLog('oc',''+e,'e');});},1200);}

function copySol(){navigator.clipboard.writeText(fullSolCode).then(function(){showToast('Verifier.sol copied!');addLog('oc','Verifier.sol copied','s');});}
function copyCalldata(){navigator.clipboard.writeText(fullCalldata).then(function(){showToast('Calldata copied!');addLog('oc','Calldata copied','s');});}
</script>
</body>
</html>"""

# ─── START FLASK + NGROK ─────────────────────────────────────────
def run_app():
    app.run(port=5008, use_reloader=False, debug=False)

t = threading.Thread(target=run_app, daemon=True)
t.start()
time.sleep(3)

# ✅ Replace with your actual token if needed
ngrok.set_auth_token("3BDmiygIQ5UYqvOGYUEUC3VguKV_5GGVByxzz3QFRYLZZpwSa")
url = ngrok.connect(5008)
print(f"\n🚀 YOUR APP IS LIVE AT: {url}")
print("✅ Open this URL in your browser!")

✅ Full model loaded
🔄 Building small model from training data...
✅ Small model trained & saved — accuracy: 75.03%
🔏 Generating small model ZK proof (this takes a minute)...
⚠️ Small model proof skipped: No module named 'onnxscript'
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5008
INFO:werkzeug:Press CTRL+C to quit



🚀 YOUR APP IS LIVE AT: NgrokTunnel: "https://kinesically-untransformative-eleonora.ngrok-free.dev" -> "http://localhost:5008"
✅ Open this URL in your browser!


In [ ]:
import joblib
import numpy as np
import torch

scaler = joblib.load('scaler.pkl')

# Normal traffic preset
normal = [0,1,49,10,181,5450,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,8,8,0,0,0,0,1,0,0,9,9,1,0,0.11,0,0,0,0,0]

# Scale it
scaled = scaler.transform(np.array(normal).reshape(1,-1))
tensor = torch.FloatTensor(scaled)

with torch.no_grad():
    output = model(tensor).item()

print(f"Raw output: {output}")
print(f"Prediction: {'MALICIOUS' if output >= 0.5 else 'NORMAL'}")

In [ ]:
import joblib
import numpy as np

scaler = joblib.load('scaler.pkl')

# Test what happens when all zeros go through scaler
zeros = [0]*41
scaled_zeros = scaler.transform(np.array(zeros).reshape(1,-1))[0].tolist()
tensor_zeros = torch.FloatTensor([scaled_zeros])
with torch.no_grad():
    out_zeros = model(tensor_zeros).item()
print(f"All zeros → scaled → output: {out_zeros:.4f} → {'MALICIOUS' if out_zeros >= 0.5 else 'NORMAL'}")

# Test normal traffic
normal = [0,1,49,10,181,5450,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,8,8,0,0,0,0,1,0,0,9,9,1,0,0.11,0,0,0,0,0]
scaled_normal = scaler.transform(np.array(normal).reshape(1,-1))[0].tolist()
tensor_normal = torch.FloatTensor([scaled_normal])
with torch.no_grad():
    out_normal = model(tensor_normal).item()
print(f"Normal traffic → scaled → output: {out_normal:.4f} → {'MALICIOUS' if out_normal >= 0.5 else 'NORMAL'}")

# Test malicious traffic
malicious = [0,2,7,7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,229,10,0,0,1,1,0.04,0.06,0,255,10,0.04,0.06,0,0,0,0,1,1]
scaled_malicious = scaler.transform(np.array(malicious).reshape(1,-1))[0].tolist()
tensor_malicious = torch.FloatTensor([scaled_malicious])
with torch.no_grad():
    out_malicious = model(tensor_malicious).item()
print(f"Malicious traffic → scaled → output: {out_malicious:.4f} → {'MALICIOUS' if out_malicious >= 0.5 else 'NORMAL'}")